# exp_18 - Tier 2A Phase A: masked-reconstruction PRETRAINING (track_c, Session 2)

Plan: docs/track_3_rl_reuse/session_plan.md §5 Session 2. Runs on **local AND Kaggle** (see the
bootstrap cell + docs/kaggle_workflow.md).

**Idea (plain):** before forecasting rain, teach the GAT-GRU encoder to *fill in the blanks*.
Hide ~15% of the physical values in each 7-day window (rain + temp/dew/pressure/u10/v10) and
reconstruct them from visible neighbours, the rest of the window, and the calendar/geography
features (never hidden). Session 3 reuses the trained encoder for the real rain forecast.

Saves `pretext_best.pt` (encoder), the reconstruction-loss curve, and a node-embedding health check.
BERT-style masking (mask 15%, encode all, predict masked); single seed 42; train period only.

In [ ]:
%matplotlib inline

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

In [ ]:
# --- environment bootstrap: identical notebook runs on BOTH local and Kaggle ---
if os.path.exists('/kaggle/working'):                       # ---- Kaggle ----
    REPO = '/kaggle/working/Dissertation'
    if not os.path.exists(REPO):
        # PUBLIC repo: plain clone. PRIVATE repo: use a PAT secret (see docs/kaggle_workflow.md).
        os.system('git clone --depth 1 https://github.com/itsRkg/Dissertation.git ' + REPO)
    KAGGLE_DATA_SLUG = 'REPLACE_WITH_YOUR_KAGGLE_DATASET_SLUG'   # e.g. 'itsrkg/dissertation-era5'
else:                                                        # ---- local ----
    REPO = 'C:/Users/rishe/Dissertation'
    KAGGLE_DATA_SLUG = None
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('repo on path:', REPO)

In [ ]:
from utils.data_utils.data_helper_utils import (
    load_pivots, get_lat_lon_aligned, build_edge_index_radius, scale_pivots, temporal_split)
from utils.data_utils.dataset_files.gnn_dataset import (
    build_feature_tensor, build_time_features, add_time_features)
from models.gat_gru_pretrain import GAT_GRU_Pretrain
from utils.metric_utils.embedding_diag import embedding_report, last_gat_embeddings
from utils.env_paths import get_paths

In [ ]:
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# BATCH=64 is a safe default for 8GB VRAM (exp_13 used 32 on this graph). OOM -> 32; headroom -> 128.
CFG = dict(epochs=30, lr=3e-4, batch=64, mask_ratio=0.15, hidden=64, heads=4,
           seq_len=7, n_recon=6, in_channels=15)
P = get_paths(kaggle_data_slug=KAGGLE_DATA_SLUG)   # resolves local vs Kaggle paths
PIVOT_PATH = P['pivot_dir']
SAVE_DIR = P['models_dir'] + '/exp_18_gat_gru_mae_pretrain/pretext'
RES_DIR  = P['results_dir'] + '/exp_18_gat_gru_mae_pretrain/pretext'
for d in (SAVE_DIR, RES_DIR): os.makedirs(d, exist_ok=True)
print(('KAGGLE' if P['is_kaggle'] else 'LOCAL'), '| data:', P['data_dir'])
print('device', device, '| cfg', CFG)

In [ ]:
# ---- data + graph + scaler fix + 15-feat tensor (identical recipe to exp_16) ----
pivots = load_pivots(PIVOT_PATH)
station_df = pd.read_csv(P['coords_csv']).rename(columns={'latitude':'lat','longitude':'lon'})
elev_df = pd.read_csv(P['elev_csv'])
lat, lon = get_lat_lon_aligned(pivots['rain'], station_df)
edge_index = build_edge_index_radius(lat, lon, threshold_km=100).to(device)
N = pivots['rain'].shape[1]
dates = pivots['rain'].index
T = len(dates); train_end_date = dates[int(0.7*T)-1]
print('train_end_date', train_end_date)
scaled_pivots, scalers = scale_pivots(pivots, train_end=train_end_date)
X_dyn, feature_order = build_feature_tensor(scaled_pivots, use_latent=True)  # 6 physical, rain=idx0
X12 = add_time_features(X_dyn, build_time_features(dates))                   # +6 cyclic time
elev_map = dict(zip(elev_df['station_id'].astype(str), elev_df['elevation_m'].astype(float)))
elev = np.array([elev_map[str(c)] for c in pivots['rain'].columns], dtype=float)
def _z(a):
    a=np.asarray(a,float); s=a.std(); return (a-a.mean())/(s if s else 1.0)
statics = np.stack([_z(lat),_z(lon),_z(elev)], axis=-1)
X15 = np.concatenate([X12, np.broadcast_to(statics[None,:,:], (X12.shape[0], N, 3))], axis=-1)
print('X15', X15.shape, '| physical channels (masked) = first', CFG['n_recon'], '=', feature_order)

In [ ]:
# ---- pretext windows: TRAIN + VAL periods only (no test; no leakage) ----
X_train, X_val, X_test = temporal_split(X15, dates)   # same 70/15/15 chronological split as exp_16
class PretextWindowDataset(Dataset):
    def __init__(self, X, seq_len):
        self.X = torch.tensor(X, dtype=torch.float32); self.seq_len = seq_len
    def __len__(self):
        return self.X.shape[0] - self.seq_len + 1
    def __getitem__(self, i):
        return self.X[i:i+self.seq_len]   # (L, N, F) ; reconstruct within the window
train_loader = DataLoader(PretextWindowDataset(X_train, CFG['seq_len']), batch_size=CFG['batch'], shuffle=True)
val_loader   = DataLoader(PretextWindowDataset(X_val,   CFG['seq_len']), batch_size=CFG['batch'])
print('train windows', len(train_loader.dataset), '| val windows', len(val_loader.dataset),
      '| steps/epoch ~', len(train_loader))

In [ ]:
# ---- masked-reconstruction pretraining (BERT-style: zero masked cells, predict them) ----
def mask_input(x, ratio, nrec, gen=None):
    phys = x[..., :nrec]
    if gen is None:
        m = (torch.rand(phys.shape, device=x.device) < ratio)
    else:
        m = (torch.rand(phys.shape, generator=gen) < ratio).to(x.device)   # fixed masks for val
    x_in = x.clone()
    x_in[..., :nrec] = torch.where(m, torch.zeros_like(phys), phys)         # 0 = neutral (z-scored)
    return x_in, phys, m

model = GAT_GRU_Pretrain(in_channels=CFG['in_channels'], hidden_dim=CFG['hidden'],
                         heads=CFG['heads'], n_recon=CFG['n_recon']).to(device)
opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
scaler = GradScaler(enabled=(device.type=='cuda'))
best_val = float('inf'); history = []; t0 = time.time()
for epoch in range(1, CFG['epochs']+1):
    model.train(); tr=0.0; ntr=0
    for x in train_loader:
        x = x.to(device)
        x_in, phys, m = mask_input(x, CFG['mask_ratio'], CFG['n_recon'])
        if m.sum()==0: continue
        opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=(device.type=='cuda')):
            rec = model(x_in, edge_index)
            loss = ((rec - phys)[m]**2).mean()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tr += loss.item()*x.size(0); ntr += x.size(0)
    tr /= max(ntr,1)
    model.eval(); vg = torch.Generator().manual_seed(12345); vl=0.0; nv=0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device)
            x_in, phys, m = mask_input(x, CFG['mask_ratio'], CFG['n_recon'], gen=vg)
            if m.sum()==0: continue
            with autocast(device_type='cuda', enabled=(device.type=='cuda')):
                rec = model(x_in, edge_index)
                loss = ((rec - phys)[m]**2).mean()
            vl += loss.item()*x.size(0); nv += x.size(0)
    vl /= max(nv,1)
    history.append(dict(epoch=epoch, train_recon_mse=tr, val_recon_mse=vl))
    print('epoch %02d | train %.5f | val %.5f | %.0fs' % (epoch, tr, vl, time.time()-t0))
    if np.isnan(tr) or np.isnan(vl):
        print('  !! NaN — stop and lower lr / batch / mask_ratio (session_plan §9)'); break
    if vl < best_val:
        best_val = vl; torch.save(model.state_dict(), SAVE_DIR + '/pretext_best.pt')
        print('  new best val %.5f -> pretext_best.pt' % vl)
torch.save(model.state_dict(), SAVE_DIR + '/pretext_last.pt')
pd.DataFrame(history).to_csv(RES_DIR + '/pretext_loss_curve.csv', index=False)
print('done. best val recon MSE = %.5f' % best_val)

In [ ]:
# ---- stability plot + node-embedding health check on the PRETRAINED encoder (reused in Tier 2A) ----
h = pd.DataFrame(history)
plt.figure(figsize=(6,4))
plt.plot(h['epoch'], h['train_recon_mse'], marker='o', label='train')
plt.plot(h['epoch'], h['val_recon_mse'], marker='o', label='val')
plt.xlabel('epoch'); plt.ylabel('masked-cell recon MSE'); plt.title('Tier 2A pretext loss'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.savefig(RES_DIR + '/pretext_loss_curve.png', dpi=120); plt.show()

model.load_state_dict(torch.load(SAVE_DIR + '/pretext_best.pt', map_location=device))
xb = next(iter(val_loader)).to(device)                 # clean (unmasked) window
emb = last_gat_embeddings(model, xb, edge_index, device)  # (B, N, D) last-GAT node embeddings
ei_np = edge_index.detach().cpu().numpy()
reps = [embedding_report(emb[b].cpu().numpy(), ei_np) for b in range(emb.shape[0])]
edrep = {k: float(np.mean([r[k] for r in reps])) for k in reps[0]}
pd.DataFrame([edrep]).to_csv(RES_DIR + '/pretext_embedding_diag.csv', index=False)
print('pretrained-encoder embedding diagnostic:'); print(pd.DataFrame([edrep]).to_string(index=False))

## How to read / what to check

**Stability gate (main goal):** `val_recon_mse` should DECREASE then flatten, no NaN. If so,
`pretext_best.pt` is ready for Session 3. If it diverges/NaNs, lower `lr` (1e-4) or `batch` (32),
or drop `mask_ratio` to 0.10 (session_plan §9).

**Embedding health:** `pretext_embedding_diag.csv` should show non-trivial `dirichlet_energy_norm`/
`MAD_cosine` and `effective_rank` well above 1 (different stations -> different embeddings). If it
collapsed toward (0,0,1), the pretext is degenerate and Session 3 transfer won't help.

**On Kaggle:** outputs are under `/kaggle/working/experiments/...` — download them from the Output
tab, or `kaggle kernels output <user>/<kernel> -p ./from_kaggle`. (Results are gitignored, so they
come back via Kaggle, not git — see docs/kaggle_workflow.md.)

**Next (Session 3):** load `pretext_best.pt`, copy `gat`+`gru` into GAT_GRU_Model(in_channels=15),
fresh rain head, finetune, compare to Tier 0b. Single seed; multi-seed only at Session 5.